In [20]:
!pip install groq

In [21]:
import os
import json
from groq import Groq
from google.colab import userdata

# Securely fetch Groq API Key
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# Fallback to the widely available model
MODEL = "openai/gpt-oss-20b"

In [24]:
def calculate(expression):
    """Evaluate a basic mathematical expression."""
    try:
        # Stripping spaces and common separators
        expression = expression.replace(',', '').replace(' ', '')
        return str(eval(expression, {"__builtins_": None}, {}))
    except Exception as e:
        return str(e)

def metadata_filter(category, value):
    """Filter mock metadata based on category and value."""
    mock_data = [
        {"id": 1, "name": "Report_A", "tag": "finance"},
        {"id": 2, "name": "Log_B", "tag": "system"},
        {"id": 3, "name": "Note_C", "tag": "finance"}
    ]
    # Normalize values for better matching
    normalized_value = value.lower().strip()
    if "financ" in normalized_value:
        normalized_value = "finance"

    results = [item for item in mock_data if item.get(category) == normalized_value]
    return json.dumps(results)

tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "ALWAYS use this tool for ANY arithmetic or math calculations, even simple ones. Input should be a mathematical string like '1234 * 5678'.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The math expression to evaluate."}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "metadata_filter",
            "description": "Filter document metadata. Available tags are 'finance' and 'system'.",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {"type": "string", "enum": ["tag", "name"], "description": "The field to filter by."},
                    "value": {"type": "string", "description": "The specific value to search for (e.g., 'finance')."}
                },
                "required": ["category", "value"]
            }
        }
    }
]

In [25]:
def run_conversation(user_prompt):
    # Adding a system message to guide the model's behavior
    messages = [
        {"role": "system", "content": "You are a helpful assistant with access to tools. If a user asks for a calculation, you MUST use the calculate tool. If searching for documents, use metadata_filter with the tag 'finance' for financial queries."},
        {"role": "user", "content": user_prompt}
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    if tool_calls:
        print(f"[System] Model decided to use tools: {[tc.function.name for tc in tool_calls]}")
        messages.append(response_message)

        available_functions = {
            "calculate": calculate,
            "metadata_filter": metadata_filter,
        }

        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)
            function_response = function_to_call(**function_args)
            print(f"[System] Tool '{function_name}' output: {function_response}")

            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": function_response,
            })

        final_response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )
        return final_response.choices[0].message.content
    else:
        print("[System] Model decided to answer directly.")
        return response_message.content

# Testing dynamic decision making
try:
    print(f"Active Model: {MODEL}")
    print("--- Test 1 (Direct) ---")
    print("Response:", run_conversation("Who won the world cup in 2022?"))

    print("\n--- Test 2 (Calculator Tool) ---")
    print("Response:", run_conversation("What is 1234 * 5678?"))

    print("\n--- Test 3 (Filter Tool) ---")
    print("Response:", run_conversation("Find all financial documents in the system."))
except Exception as e:
    print(f"Execution Error: {e}")

Active Model: openai/gpt-oss-20b
--- Test 1 (Direct) ---
[System] Model decided to answer directly.
Response: Argentina won the 2022 FIFA World Cup, defeating France in a dramatic final that ended 3‑3 after extra time and was decided by a penalty shootout (4‑2).

--- Test 2 (Calculator Tool) ---
[System] Model decided to use tools: ['calculate']
[System] Tool 'calculate' output: 7006652
Response: 1234 × 5678 = **7,006,652**

--- Test 3 (Filter Tool) ---
[System] Model decided to use tools: ['metadata_filter']
[System] Tool 'metadata_filter' output: [{"id": 1, "name": "Report_A", "tag": "finance"}, {"id": 3, "name": "Note_C", "tag": "finance"}]
Response: Here are the financial documents found in the system:

| ID | Name      | Tag     |
|----|-----------|---------|
| 1  | Report_A  | finance |
| 3  | Note_C    | finance |

Let me know if you’d like any additional details about these documents.
